In [11]:
from pathlib import Path

ROOT = Path(".").resolve().parents[1]
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [12]:
from rich import print as rprint

In [13]:
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [25]:
from src.application.contracts import PipelineRequest
from src.application import ViRAGEPipeline
from src.application.settings import ViRAGESettings

tmp_path = Path("../demo_data/tmp_folder").resolve()
data_path = Path("../demo_data/Iris.csv").resolve()
settings = ViRAGESettings(artifact_root=tmp_path / "artifacts")
pipeline = ViRAGEPipeline(settings)
query="Show the sales trend over time"
request = PipelineRequest(query=query, data_path=data_path.as_posix())

In [26]:
from langchain_ollama import ChatOllama

LLM_MODEL = "gemma3:1b"  # или "llama3.2:1b"
# LLM_MODEL = "llama3.2:1b"       # или "llama3.2:1b"
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0,
)

In [27]:
from src.infrastructure import RuntimeContext

runtime = RuntimeContext(settings=settings, llm=llm)

In [28]:
from src.services import QueryUnderstandingService

qu = QueryUnderstandingService().invoke(runtime=runtime, user_context=request.user_context, query=request.query)

In [29]:
rprint("query:", request.query)
rprint(qu)

query: Show the sales trend over time

QueryUnderstandingResult(
    intent='Show the sales trend over time',
    requested_operations=['trend analysis'],
    candidate_charts=['line'],
    constraints=[],
    case_type=<ChartCaseType.CANONICAL: 'canonical'>,
    confidence=0.8
)

In [30]:
from src.services import CanonicalPlanningService

cp = CanonicalPlanningService().invoke(runtime=runtime, query_understanding=qu)

In [31]:
rprint(cp)

PlanningResult(
    mode=<ChartCaseType.CANONICAL: 'canonical'>,
    steps=[
        PlanningStep(
            name='profile_the_dataset_and_confirm_field',
            description='Profile the dataset and confirm field types relevant to the request.'
        ),
        PlanningStep(
            name='prepare_a_cleaned_analysis_ready_version',
            description='Prepare a cleaned analysis-ready version of the data.'
        ),
        PlanningStep(
            name='retrieve_concise_charting_guidance_for_the',
            description='Retrieve concise charting guidance for the selected chart family.'
        ),
        PlanningStep(
            name='build_the_primary_requested_chart_using',
            description='Build the primary requested chart using the leading chart family: line.'
        ),
        PlanningStep(
            name='execute_plotting_code_and_collect_numeric',
            description='Execute plotting code and collect numeric summaries from the run.'
        ),
        PlanningStep(
            name='read_chart_structure,_extract_facts_and',
            description='Read chart structure, extract facts and verify that final statements are evidence-backed.'
        )
    ],
    success_criteria=[
        'At least one valid canonical chart is produced, preferably among: line.',
        'Generated charts are readable and consistent with the request.',
        'Final statements reference execution metrics or chart evidence.'
    ]
)

In [32]:
from src.services import DataProfilerService

data_profile = DataProfilerService().invoke(runtime=runtime, data_path=data_path)

In [33]:
rprint(data_profile)

DataProfile(
    row_count=150,
    col_count=6,
    columns=[
        DataColumnProfile(name='Id', dtype='numeric', missing_ratio=0.0, unique_count=150),
        DataColumnProfile(name='SepalLengthCm', dtype='numeric', missing_ratio=0.0, unique_count=35),
        DataColumnProfile(name='SepalWidthCm', dtype='numeric', missing_ratio=0.0, unique_count=23),
        DataColumnProfile(name='PetalLengthCm', dtype='numeric', missing_ratio=0.0, unique_count=43),
        DataColumnProfile(name='PetalWidthCm', dtype='numeric', missing_ratio=0.0, unique_count=22),
        DataColumnProfile(name='Species', dtype='categorical', missing_ratio=0.0, unique_count=3)
    ],
    likely_numeric_columns=['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm'],
    likely_categorical_columns=['Species'],
    likely_time_columns=[],
    quality_notes=["Column 'Id' looks like an identifier."]
)

In [34]:
from src.services import DataPreparationService

data_prep = DataPreparationService().invoke(runtime=runtime, data_path=data_path, data_profile=data_profile, run_id="1")

In [35]:
rprint(data_prep)

DataPreparationResult(
    output_path='D:/programming/projects/mas_rag/demo_data/tmp_folder/artifacts/1/cleaned_data.csv',
    operations=[],
    row_count=150,
    col_count=6
)

# VisRAG

In [36]:
from src.services import VisRAGService

recommendations = VisRAGService().invoke(runtime=runtime, data_profile=data_profile, query_understanding=qu)

In [37]:
rprint(recommendations)

VisRAGResult(
    recommendations=[
        VisRAGRecommendation(
            chart_family='line',
            rationale='Line chart kept as a general-purpose fallback for trend-like queries.',
            priority=1,
            score=0.55,
            support_examples=[]
        )
    ],
    rules=[
        'Use clear titles and axis labels.',
        'Avoid overcrowded visuals.',
        'Prefer readable defaults.',
        'Prefer examples retrieved from the local corpus when they agree with the data shape.'
    ],
    caveats=['No matching local corpus examples were retrieved; heuristic ranking was used.'],
    retrieved_examples=[],
    corpus_status={'plot2code': 'not_configured'},
    retrieval_strategy='heuristic_only'
)

In [40]:
from pathlib import Path

from src.application.settings import ViRAGESettings
from src.infrastructure.runtime import RuntimeContext
from src.services.visrag import VisRAGService

runtime = RuntimeContext(
    settings=ViRAGESettings(
        artifact_root=Path("./artifacts").resolve(),
        visrag_corpus_root=Path("../data/visrag_corpora").resolve(),
        visrag_top_k_examples=5,
        visrag_top_k_recommendations=3,
        visrag_min_example_score=0.05,
        llm=llm
    )
)

result = VisRAGService().invoke(qu, data_profile, runtime=runtime)
rprint(result.model_dump_json(indent=2, ensure_ascii=False))

{
  "recommendations": [
    {
      "chart_family": "line",
      "rationale": "Line chart kept as a general-purpose fallback for trend-like queries. 5 local corpus match(es) 
from Plot2Code support this chart family.",
      "priority": 1,
      "score": 1.4967,
      "support_examples": [
        "plot2code-175764f4bb98",
        "plot2code-3b2021d3ad70",
        "plot2code-7f3361de2144"
      ]
    }
  ],
  "rules": [
    "Use clear titles and axis labels.",
    "Avoid overcrowded visuals.",
    "Prefer readable defaults.",
    "Prefer examples retrieved from the local corpus when they agree with the data shape."
  ],
  "caveats": [],
  "retrieved_examples": [
    {
      "example_id": "plot2code-175764f4bb98",
      "source": "Plot2Code",
      "chart_type": "line",
      "instruction": "1. Type of Figure: The figure is a side-by-side comparison of two line plots.\n\n2. Data: The
data used for both plots is a range of numbers from 0.0 to 5.0, with 501 points in between. The y-values for the 
first plot are calculated by multiplying the cosine of 6 times each x-value by the exponential of the negative of 
each x-value. The y-values for the second plot are simply the cosine of 6 times each x-value.\n\n3. Titles: The 
first plot is titled 'damped' and the second plot is titled 'undamped'. The overall figure has a title 'Different 
types of oscillations' in a font size of 16.\n\n4. Axes: Both plots share the same y-axis. The x-axis is labeled 
'time (s)' for both plots, and the y-axis (shared) is labeled 'amplitude'.\n\n5. Layout: The layout of the subplots
is 'constrained', meaning they are arranged in a way that prevents overlapping.\n\n6. Other Details: The two plots 
are created in a single row, with the 'damped' plot on the left and the 'undamped' plot on the right.",
      "description": null,
      "tags": [
        "line",
        "trend analysis",
        "comparison"
      ],
      "code_language": "python",
      "domain": "general",
      "score": 0.9467,
      "rationale": "lexical_overlap=0.42; preferred_chart_bias=0.35; time_or_trend_match"
    },
    {
      "example_id": "plot2code-3b2021d3ad70",
      "source": "Plot2Code",
      "chart_type": "line",
      "instruction": "The figure generated from the provided Python code consists of two subplots arranged 
vertically. \n\nThe first subplot is a line graph that plots two signals over time. These signals are composed of a
coherent part at 10 Hz and a random part. The x-axis represents time in seconds, ranging from 0 to 2 seconds. The 
y-axis represents the two signals, labeled as 's1' and 's2'. The graph also includes a grid for easier reading of 
the data points.\n\nThe second subplot is a coherence plot of the two signals. The y-axis is labeled as 
'Coherence'. The coherence function is calculated with a window size of 256 and a noverlap of 1/dt.\n\nThe random 
state is fixed at 19680801 for reproducibility. This ensures that the random noise added to the signals is the same
every time the code is run. \n\nTo recreate this figure, you would need to generate two signals with a coherent 
part at 10 Hz and a random part, plot them over time in the first subplot, and calculate and plot their coherence 
in the second subplot. You would also need to set the random state to 19680801 to ensure the random noise is the 
same.",
      "description": null,
      "tags": [
        "line",
        "trend analysis"
      ],
      "code_language": "python",
      "domain": "general",
      "score": 0.9467,
      "rationale": "lexical_overlap=0.42; preferred_chart_bias=0.35; time_or_trend_match"
    },
    {
      "example_id": "plot2code-7f3361de2144",
      "source": "Plot2Code",
      "chart_type": "line",
      "instruction": "The figure generated by the provided Python code consists of two subplots sharing the same 
x-axis. The first subplot is a line plot representing a signal over time, while the second subplot is a spectrogram
of the same signal.\n\nThe signal is cr